# Study and prepare artist names

We inspect surrounding whitespace in `artist_name`, compare the follower counts of the affected names, and remove leading and trailing spaces. We preserve internal spaces, capitalization, accents, missing values, followers, and every other column.

The cleaned copies remain in memory as `train_clean` and `test_clean`. We record the analysis and decision in `reports/artist_name_preparation/summary.md`. Raw CSV files and the existing genre-only processed CSV remain unchanged.

## 1. Load the original datasets

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

DATA_DIR = next(
    (directory for directory in (
        Path.cwd() / "data/raw",
        Path.cwd().parent / "data/raw",
        Path.cwd(),
    ) if all((directory / name).is_file() for name in ("train.csv", "test.csv"))),
    None,
)
if DATA_DIR is None:
    raise FileNotFoundError("Set DATA_DIR to the folder containing train.csv and test.csv.")
train_raw = pd.read_csv(DATA_DIR / "train.csv")
test_raw = pd.read_csv(DATA_DIR / "test.csv")


## 2. Study whitespace and follower consistency

We examine the affected names across both datasets. The printed original names use `repr` so trailing spaces are visible. Follower consistency is a metadata check, not proof of artist identity.

In [2]:
def artist_name_whitespace_mask(values):
    return values.notna() & values.ne(values.str.strip()).fillna(False)


def summarize_artist_whitespace(datasets):
    return pd.DataFrame([
        {
            "dataset": name,
            "rows with surrounding spaces": int(artist_name_whitespace_mask(frame["artist_name"]).sum()),
            "distinct affected names": frame.loc[
                artist_name_whitespace_mask(frame["artist_name"]), "artist_name"
            ].nunique(),
        }
        for name, frame in datasets.items()
    ])


def trim_artist_names(frame):
    cleaned = frame.copy()
    cleaned["artist_name"] = cleaned["artist_name"].str.strip()
    return cleaned

artist_whitespace_summary = summarize_artist_whitespace({"train": train_raw, "test": test_raw})
display(artist_whitespace_summary)

artist_names = pd.concat([
    train_raw[["artist_name", "artist_followers"]],
    test_raw[["artist_name", "artist_followers"]],
], ignore_index=True)
artist_names["trimmed_name"] = artist_names["artist_name"].str.strip()
changed_names = artist_names.loc[
    artist_name_whitespace_mask(artist_names["artist_name"]), "trimmed_name"
].drop_duplicates()
artist_name_examples = []
for name in sorted(changed_names):
    group = artist_names[artist_names["trimmed_name"].eq(name)]
    artist_name_examples.append({
        "original names (repr)": ", ".join(repr(value) for value in sorted(group["artist_name"].unique())),
        "trimmed name": name,
        "tracks across train and test": len(group),
        "distinct follower counts": group["artist_followers"].nunique(),
        "follower counts": ", ".join(str(value) for value in sorted(group["artist_followers"].unique())),
        "rows already without spaces": int(group["artist_name"].eq(name).sum()),
    })
display(pd.DataFrame(artist_name_examples))


,dataset,rows with surrounding spaces,distinct affected names
0,train,34,2
1,test,9,2


,original names (repr),trimmed name,tracks across train and test,distinct follower counts,follower counts,rows already without spaces
0,'Raven Black ',Raven Black,11,1,21127,0
1,'Workout Music ',Workout Music,32,1,72649,0


## 3. Decision and transformation

We found 34 affected training rows and 9 affected test rows. The two affected names, `Workout Music ` and `Raven Black `, have consistent follower counts within each name. Neither has an existing version without the trailing space.

We standardize surrounding spaces with `str.strip()` in both datasets. We do not merge artists, alter internal spaces, or propagate genre information.

In [3]:
train_clean = trim_artist_names(train_raw)
test_clean = trim_artist_names(test_raw)
for original, cleaned in ((train_raw, train_clean), (test_raw, test_clean)):
    pd.testing.assert_frame_equal(original.drop(columns="artist_name"), cleaned.drop(columns="artist_name"))
    pd.testing.assert_series_equal(original["artist_name"].str.strip(), cleaned["artist_name"])
    assert not artist_name_whitespace_mask(cleaned["artist_name"]).any()
    assert cleaned["artist_name"].isna().equals(original["artist_name"].isna())
display(summarize_artist_whitespace({"train_clean": train_clean, "test_clean": test_clean}))


,dataset,rows with surrounding spaces,distinct affected names
0,train_clean,0,0
1,test_clean,0,0


## 4. Record our decision

In [4]:
report_dir = DATA_DIR.parent.parent / "reports/artist_name_preparation"
report_dir.mkdir(parents=True, exist_ok=True)
lines = [
    "# Artist name whitespace preparation", "",
    "We examined surrounding whitespace in `artist_name` and compared follower counts for the affected names.", "",
    "| Dataset | Rows with surrounding spaces | Distinct affected names |",
    "| --- | ---: | ---: |",
]
for _, row in artist_whitespace_summary.iterrows():
    lines.append(f"| {row['dataset']} | {row['rows with surrounding spaces']} | {row['distinct affected names']} |")
lines += [
    "", "| Original name | Trimmed name | Tracks across train and test | Follower counts | Existing rows without spaces |",
    "| --- | --- | ---: | --- | ---: |",
]
for row in artist_name_examples:
    lines.append(
        f"| `{row['original names (repr)']}` | {row['trimmed name']} | "
        f"{row['tracks across train and test']} | {row['follower counts']} | {row['rows already without spaces']} |"
    )
lines += [
    "", "## Decision", "",
    "We removed leading and trailing whitespace from `artist_name` in copies of both datasets. "
    "We preserved internal spaces, capitalization, accents, missing values, followers, and all other columns. "
    "The affected names have consistent follower counts, and neither name has an existing version without surrounding spaces. "
    "This operation therefore standardizes the names without merging existing name variants. "
    "We did not infer artist identity or fill genre information from names.", "",
    "We included the same transformation in `delivery/music_hit_prediction.ipynb`. "
    "Our working notebook keeps the cleaned frames in memory and leaves the raw CSV files unchanged.", "",
]
(report_dir / "summary.md").write_text("\n".join(lines), encoding="utf-8")
print(f"Report saved to {report_dir / 'summary.md'}")


Report saved to /home/goncalotorcato/ist1109358/CD/Projeto/CD/reports/artist_name_preparation/summary.md
